In [1]:
import pandas as pd
import os
import numpy as np

## Hechos

In [4]:
archivos=os.listdir("data/estado")
atributos=["station_id","num_bikes_available","num_bikes_available_types.mechanical","num_bikes_available_types.ebike","num_docks_available","last_reported","status"]
hechos=pd.concat([pd.read_csv(f"data/estado/{archivo}",usecols=atributos,low_memory=False).drop_duplicates() for archivo in archivos],axis=0)

In [12]:
hechos[hechos.status=="IN_SERVICE"].head()

,station_id,num_bikes_available,num_bikes_available_types.mechanical,num_bikes_available_types.ebike,num_docks_available,last_reported,status
0,1.0,38.0,38.0,0.0,6.0,1.593554e+09,IN_SERVICE
1,2.0,1.0,1.0,0.0,20.0,1.593554e+09,IN_SERVICE
2,3.0,20.0,20.0,0.0,5.0,1.593554e+09,IN_SERVICE
3,4.0,2.0,2.0,0.0,19.0,1.593554e+09,IN_SERVICE
4,5.0,30.0,30.0,0.0,9.0,1.593554e+09,IN_SERVICE


In [9]:
hechos.status.value_counts(dropna=False)

status
IN_SERVICE        272350390
NOT_IN_SERVICE       156877
MAINTENANCE          147580
PLANNED                5440
NaN                      19
Name: count, dtype: int64

In [7]:
hechos.isnull().sum()

station_id                              19
num_bikes_available                     19
num_bikes_available_types.mechanical    19
num_bikes_available_types.ebike         19
num_docks_available                     19
last_reported                           37
status                                  19
dtype: int64

In [21]:
hechos=hechos.dropna(subset=['station_id','last_reported'])

In [22]:
hechos=hechos.rename(columns={
    'num_bikes_available_types.mechanical': 'num_bikes_available_mechanical',
    'num_bikes_available_types.ebike': 'num_bikes_available_ebike'
    }
    )

In [23]:
#Corregir el formato
hechos=hechos.assign(
    station_id=hechos['station_id'].astype(int),
    num_bikes_available=hechos['num_bikes_available'].astype(int),
    num_bikes_available_mechanical=hechos['num_bikes_available_mechanical'].astype(int),
    num_bikes_available_ebike=hechos['num_bikes_available_ebike'].astype(int),
    num_docks_available=hechos['num_docks_available'].astype(int),
    last_reported=pd.to_datetime(hechos['last_reported'], unit='s')
)


In [24]:
hechos.sample(10)

,station_id,num_bikes_available,num_bikes_available_mechanical,num_bikes_available_ebike,num_docks_available,last_reported
318054,448,6,6,0,17,2023-10-03 01:57:09
2096875,253,6,2,4,18,2025-05-15 02:53:32
3516299,21,14,2,12,7,2023-02-25 04:39:19
462838,413,3,3,0,28,2023-07-04 01:51:09
4527728,424,23,22,1,0,2025-03-31 16:39:49
2029696,440,7,2,5,19,2024-08-15 10:47:54
1809667,21,17,17,0,2,2023-12-13 09:41:20
145498,62,0,0,0,25,2022-11-01 22:57:04
3629823,456,4,1,3,13,2022-06-25 18:33:35
1583398,313,8,1,7,21,2025-09-11 00:32:28


In [22]:
hechos=pd.read_csv("docs/entregas/estado/2025_07_Juliol_BicingNou_ESTACIONS.csv",low_memory=False)
hechos.sample(5)

,station_id,num_bikes_available,num_bikes_available_types.mechanical,num_bikes_available_types.ebike,num_docks_available,last_reported,is_charging_station,status,is_installed,is_renting,is_returning,traffic,last_updated,ttl
2405558,237,7,5,2,16,1.752677e+09,True,IN_SERVICE,1,1,1,NaN,1752677400,0
614280,243,4,4,0,22,1.751667e+09,True,IN_SERVICE,1,1,1,NaN,1751667305,0
3909007,83,5,5,0,16,1.753516e+09,True,IN_SERVICE,1,1,1,NaN,1753516203,0
1383069,541,1,0,1,26,1.752102e+09,True,IN_SERVICE,1,1,1,NaN,1752102300,0
3683955,493,0,0,0,22,1.753391e+09,True,IN_SERVICE,1,1,1,NaN,1753391400,0


In [23]:
hechos.dtypes

station_id                                int64
num_bikes_available                       int64
num_bikes_available_types.mechanical      int64
num_bikes_available_types.ebike           int64
num_docks_available                       int64
last_reported                           float64
is_charging_station                        bool
status                                      str
is_installed                              int64
is_renting                                int64
is_returning                              int64
traffic                                 float64
last_updated                              int64
ttl                                       int64
dtype: object

In [24]:
sum(hechos.last_reported<hechos.last_updated)

4770553

### Dimensiones

In [2]:
files=sorted(os.listdir("data/informacion/"))

In [3]:
atributos = [
    "station_id",
    "physical_configuration",
    "lat",
    "lon",
    "address",
    "post_code",
    "capacity",
    "last_updated"
]

In [7]:
dimensiones=pd.concat([pd.read_csv("docs/entregas/informacion/"+ file, usecols=atributos, dtype={"post_code": str}, low_memory=False, encoding="latin-1") for file in files],
axis=0, ignore_index=True)
dimensiones = dimensiones.dropna(subset=["station_id"])
dimensiones=dimensiones.drop_duplicates(subset=["station_id"], keep="last")
dimensiones=dimensiones.rename(columns={"lat": "latitud", "lon": "longitud"})


In [16]:
dimensiones.head()

,station_id,physical_configuration,latitud,longitud,address,post_code,capacity,last_updated
1000043,93.0,ELECTRICBIKESTATION,41.375632,2.149669,"GRAN VIA DE LES CORTS CATALANES, 375-385",08015,24.0,1.594173e+09
20938020,530.0,VALET,41.347855,2.119410,MercÃ¨,00001,1.0,1.607012e+09
57923195,529.0,VALET,41.347768,2.119389,Nassos,00001,1.0,1.629109e+09
119056331,532.0,ELECTRICBIKESTATION,41.357937,2.127649,2198 Avinguda de la Granvia de lâHospitalet,08908,1.0,1.668324e+09
196622513,521.0,ELECTRICBIKESTATION,41.378461,2.157360,"C/ Calabria, 66",08015,2.0,1.715703e+09


In [12]:
dimensiones.dtypes

station_id                float64
physical_configuration        str
latitud                   float64
longitud                  float64
address                       str
post_code                     str
capacity                  float64
last_updated              float64
dtype: object

In [8]:
def formato_cp(cp):
    """
    Formatea el código postal para que tenga 5 dígitos.
    Si no es un string numérico, devuelve np.nan.
    """
    return np.nan if not isinstance(cp, str) else cp.zfill(5)
    

In [9]:
dimensiones["post_code"] = dimensiones["post_code"].apply(formato_cp)

In [15]:
dimensiones.to_csv("dimensiones.csv", index=False)

In [8]:
dimensiones.dtypes

station_id                  int64
physical_configuration        str
latitud                   float64
longitud                  float64
address                       str
post_code                     str
capacity                    int64
dtype: object

In [13]:
dimensiones.isna().sum()

station_id                 0
physical_configuration     3
latitud                    0
longitud                   0
address                    0
post_code                 11
capacity                   0
last_updated               0
dtype: int64

In [ ]:
columnas=["station_id","name","physical_configuration","lat","lon","address","post_code","capacity"]
archivos=os.listdir("docs/entregas/inf")

In [ ]:
address.to_csv("address.csv", index=False)

#### Estudio de capacidades a lo largo del tiempo

In [3]:
capacidades=pd.concat([pd.read_csv("data/informacion/"+ file, usecols=["station_id","capacity"], low_memory=False,encoding='latin-1') for file in files], axis=0, ignore_index=True)

In [4]:
capacidades=capacidades.dropna()

In [5]:
capacidades= capacidades.assign(
    station_id= capacidades.station_id.astype(int),
    capacity= capacidades.capacity.astype(int)
)

In [6]:
capacidades.sample(10)

,station_id,capacity
10163089,106,20
20038234,381,41
268728916,147,33
176470396,220,27
6429131,326,32
108940577,227,18
269938408,310,27
182663360,396,32
58266987,55,19
66570515,301,27


In [7]:
capacidades = capacidades[capacidades['capacity'] != 0]

In [ ]:
analisis_cap=(
    capacidades.groupby('station_id').agg(lambda x: "|"
    .join(map(str, sorted(x.unique()))))
    .reset_index()
    )

In [12]:
analisis_cap.head()

,station_id,capacity
0,1,44|45|46|47
1,2,27|28|29
2,3,23|25|26|27
3,4,20|21
4,5,19|24|26|38|39


In [13]:
analisis_cap.to_csv("analisis_cap.csv", index=False)